In [ ]:
import sys
from pathlib import Path

# Jupyter runs notebooks as top-level scripts, so relative imports don't work.
# Add python-labs/ (the parent of my_pipeline/) to sys.path so absolute imports work.
_root = Path.cwd().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import pandas as pd
from config import CONFIG
from utils import logger

In [ ]:
def ingest_website_csv(filepath: Path) -> pd.DataFrame:
    logger.info(f"Ingesting website CSV: {filepath}")
    df = pd.read_csv(
        filepath,
        encoding="iso-8859-1",
        dtype={"Phone": str},
        parse_dates=["Registration Date"],
        na_values=["", "N/A", "null", "NULL", "none", "NaN"],
    )
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^\w]", "_", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
    )
    df = df.rename(columns={
        "customeremail": "email",
        "first_name": "first_name",
        "last_name": "last_name",
        "registration_date": "registration_date",
        "optout": "opt_out",
    })
    test_mask = df["email"].str.contains(r"@test\.shopstream\.com$", na=False, case=False)
    removed = test_mask.sum()
    df = df[~test_mask].copy()
    logger.info(f"  Removed {removed} test accounts")
    df["source"] = "website"
    logger.info(f"  Ingested {len(df)} records from website CSV")
    return df

# data_work = ingest_website_csv(CONFIG["input_dir"] / "website_customers.csv")
# print(data_work)


def ingest_erp_fixed_width(filepath: Path) -> pd.DataFrame:
    logger.info(f"Ingesting ERP fixed-width: {filepath}")
    colspecs = [
        (0, 10), (10, 60), (60, 120), (120, 140), (140, 145), (145, 155), (155, 160)
    ]
    col_names = ["customer_id", "full_name", "email", "phone", "region_code", "registration_date", "status"]
    df = pd.read_fwf(filepath, colspecs=colspecs, names=col_names, dtype=str, encoding="iso-8859-1")
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip()
    name_split = df["full_name"].str.split(n=1, expand=True)
    df["first_name"] = name_split[0] if 0 in name_split.columns else None
    df["last_name"] = name_split[1] if 1 in name_split.columns else None
    df["registration_date"] = pd.to_datetime(df["registration_date"], errors="coerce")
    df["region"] = df["region_code"]
    df["source"] = "erp"
    logger.info(f"  Ingested {len(df)} records from ERP")
    return df

data_erp = ingest_erp_fixed_width(CONFIG["input_dir"] / "erp_customers.txt")
# print(data_erp.loc[0:10])


2026-05-25 09:01:06,629 [INFO] Ingesting ERP fixed-width: C:\Users\MichaelAcheampong\Desktop\python-labs\my_pipeline\data\raw\erp_customers.txt
C:\Users\MichaelAcheampong\AppData\Local\Temp\ipykernel_11736\4114892161.py:44: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:
C:\Users\MichaelAcheampong\AppData\Local\Temp\ipykernel_11736\4114892161.py:49: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify 

   customer_id         full_name                    email              phone  \
0            0       Maria Smith    customer606@gmail.com  +1 (555) 123-4567   
1            1      José Johnson  customer455@company.com   +44 20 7946 0958   
2            2    André Williams    customer501@gmail.com   +44 20 7946 0958   
3            3         Léa Brown     customer24@gmail.com         5551234567   
4            4    François Jones   customer11@company.com   +44 20 7946 0958   
5            5     Müller Garcia  customer434@company.com      020 7946 0958   
6            6  O'Brien Martínez    customer195@gmail.com         5551234567   
7            7         John Díaz     customer85@yahoo.com         5551234567   
8            8        Jane López    customer790@yahoo.com      invalid-phone   
9            9     Mike González  customer515@company.com       555.123.4567   
10          10        Sarah Wang  customer104@company.com         5551234567   

   region_code registration_date status